# **HADDOCK3 tutorial on protein-DNA docking**

# **Introduction**

This tutorial demonstrates how to setup a Haddock3 workflow dedicated to predict the 3D structure of protein-DNA (double-stranded DNA) complexes using pre-defined restraints, derived from the literature, and symmetry restraints. Here, we introduce the basic concepts of HADDOCK, suitable for tackling various typical protein-DNA docking challenges:

* basic preparation of the input PDB files;
* creation of the suitable Haddock3 workflow;
* basic analysis of the docking results.

We will work with the phage 434 Cro/OR1 complex (PDB: [3CRO](https://www.rcsb.org/structure/3CRO)), formed by bacteriophage 434 Cro repressor proteins and the OR1 operator.

Cro is part of the bacteriophage 434 genetic switch, playing a key role in controlling the switch between the lysogenic and lytic cycles of the bacteriophage. It is a repressor protein that works in opposition to the phage repressor cI protein to control the genetic switch. Both repressors compete to gain control over an operator region containing three operators that determine the state of the lytic/lysogenic genetic switch. If Cro prevails, the late genes of the phage will be expressed, resulting in lysis. Conversely, if the cI repressor prevails, the transcription of Cro genes is blocked, and cI repressor synthesis is maintained, resulting in a state of lysogeny.

The structure of the phage 434 Cro/OR1 complex was solved by X-RAY crystallography at 2.5Å. We will use this experimentally solved structure (PDBID: 3CRO) as a reference within the tutorial. Cro is a symmetrical dimer, each subunit contains a helix-turn-helix (HTH), with helices α2 and α3 being separated by a short turn. This is a DNA binding motif that is known to bind major grooves. Helix α3 is the recognition helix that fits into the major groove of the operator DNA and is oriented with its axes parallel to the major groove. The side chains of each helix are thus positioned to interact with the edges of base pairs on the floor of the groove. Non-specific interactions also help to anchor Cro to the DNA. These include H-bonds between main chain NH groups and phosphate oxygens of the DNA in the region of the operator. Cro distorts the normal B-form DNA conformation: the OR1 DNA is bent (curved) by Cro, and the middle region of the operator is overwound, as reflected in the reduced distance between phosphate backbones in the minor groove.



<figure style="text-align: center;">
  <img
src="https://www.bonvinlab.org/education/HADDOCK3/HADDOCK3-protein-DNA-basic/CRO-OR1.png">
</figure>


---
# **A brief introduction to HADDOCK3**

HADDOCK3 is the next generation integrative modelling software in the
long-lasting HADDOCK project. It represents a complete rethinking and rewriting
of the HADDOCK2.X series, implementing a new way to interact with HADDOCK and
offering new features to users who can now define custom workflows.

In the previous HADDOCK2.x versions, users had access to a highly
parameterisable yet rigid simulation pipeline composed of three steps:
`rigid-body docking (it0)`, `semi-flexible refinement (it1)`, and `final refinement (itw)`.

<figure style="text-align: center;">
<img width="75%" src="https://bonvinlab.org/education/HADDOCK3/HADDOCK3-antibody-antigen/HADDOCK2-stages.png">
</figure>

In HADDOCK3, users have the freedom to configure docking workflows into
functional pipelines by combining the different HADDOCK3 modules, thus
adapting the workflows to their projects. HADDOCK3 has therefore developed to
truthfully work like a puzzle of many pieces (simulation modules) that users can
combine freely. To this end, the old HADDOCK machinery has been modularized,
and several new modules added, including third-party software additions. As a
result, the modularization achieved in HADDOCK3 allows users to duplicate steps
within one workflow (e.g., to repeat twice the `it1` stage of the HADDOCK2.x
rigid workflow).

Note that, for simplification purposes, at this time, not all functionalities of
HADDOCK2.x have been ported to HADDOCK3, which does not (yet) support NMR RDC,
PCS and diffusion anisotropy restraints, cryo-EM restraints and coarse-graining.
Any type of information that can be converted into ambiguous interaction
restraints can, however, be used in HADDOCK3, which also supports the
*ab initio* docking modes of HADDOCK.

<figure style="text-align: center;">
<img width="75%" src="https://bonvinlab.org/education/HADDOCK3/HADDOCK3-antibody-antigen/HADDOCK3-workflow-scheme.png">
</figure>

To keep HADDOCK3 modules organized, we catalogued them into several
categories. However, there are no constraints on piping modules of different
categories.

The main module categories are "topology", "sampling", "refinement",
"scoring", and "analysis". There is no limit to how many modules can belong to a
category. Modules are added as developed, and new categories will be created
if/when needed. You can access the HADDOCK3 documentation page for the list of
all categories and modules. Below is a summary of the available modules:

* **Topology modules**
    * `topoaa`: *generates the all-atom topologies for the CNS engine.*
* **Sampling modules**
    * `rigidbody`: *Rigid body energy minimization with CNS (`it0` in haddock2.x).*
    * `lightdock`: *Third-party glow-worm swam optimization docking software.*
* **Model refinement modules**
    * `flexref`: *Semi-flexible refinement using a simulated annealing protocol through molecular dynamics simulations in torsion angle space (`it1` in haddock2.x).*
    * `emref`: *Refinement by energy minimisation (`itw` EM only in haddock2.4).*
    * `mdref`: *Refinement by a short molecular dynamics simulation in explicit solvent (`itw` in haddock2.X).*
* **Scoring modules**
    * `emscoring`: *scoring of a complex performing a short EM (builds the topology and all missing atoms).*
    * `mdscoring`: *scoring of a complex performing a short MD in explicit solvent + EM (builds the topology and all missing atoms).*
* **Analysis modules**
    * `alascan`: *Performs a systematic (or user-define) alanine scanning mutagenesis of interface residues.*
    * `caprieval`: *Calculates CAPRI metrics (i-RMSD, l-RMSD, Fnat, DockQ) with respect to the top-scoring model or reference structure if provided.*
    * `clustfcc`: *Clusters models based on the fraction of common contacts (FCC)*
    * `clustrmsd`: *Clusters models based on pairwise RMSD matrix calculated with the `rmsdmatrix` module.*
    * `contactmap`: *Generate contact matrices of both intra- and intermolecular contacts and a chordchart of intermolecular contacts.*
    * `rmsdmatrix`: *Calculates the pairwise RMSD matrix between all the models generated in the previous step.*
    * `ilrmsdmatrix`: *Calculates the pairwise interface-ligand-RMSD (il-RMSD) matrix between all the models generated in the previous step.*
    * `seletop`: *Selects the top N models from the previous step.*
    * `seletopclusts`: *Selects the top N clusters from the previous step.*

The HADDOCK3 workflows are defined in simple configuration text files, similar to the TOML format but with extra features.
Contrary to HADDOCK2.X which follows a rigid (yet highly parameterisable)
procedure, in HADDOCK3, you can create your own simulation workflows by
combining a multitude of independent modules that perform specialized tasks.



---
# **Software and data setup**

In order to follow this tutorial we will start Jupyter session in Colab or other resources, install haddock3 and download the data for the tutorial.

To run this tutorial on Colab we recommend changing the runtime type to use one of the TPU option, which will give you access to more cores. The HADDOCK3 docking run described in this tutorial should complete in about 15 minutes.

For this follow the following steps (comment out the `pip install haddock3` line if you already have a working installation):

In [ ]:
#@title 1. Install haddock3, BioPython and py3Dmol
#@markdown Execute this cell to download the required packages.
!pip install haddock3==2026.7.0 --quiet
!pip install py3Dmol --quiet
!pip install BioPython --quiet

In [ ]:
#@title 1. Download tutorial data and setup environment
#@markdown Execute this cell to download the tutorial data.
!wget https://surfdrive.surf.nl/public.php/dav/files/KSH3JDNjAjdErB6 -O HADDOCK3-protein-dna-notebook.zip
!unzip -oq HADDOCK3-protein-dna-notebook.zip
!\rm HADDOCK3-protein-dna-notebook.zip
import os
from pathlib import Path
BASE_DIR = Path.cwd()
PROJECT_NAME = 'HADDOCK3-protein-dna-notebook'
PROJECT_DIR = BASE_DIR / PROJECT_NAME
os.chdir(PROJECT_DIR)

Decompressing the file will create the `HADDOCK3-protein-dna-basic` directory with the following subdirectories and items:

* `protein-dna-basic.cfg`, a HADDOCK3 configuration file;
* `pdbs`:
    * `1ZUG.pdb`, the NMR ensemble (20 structures) of the 343 Cro repressor structures;
    * `1ZUG_dimer1.pdb`, a single structure coming from the NMR ensemble (1ZUG.pdb) with the terminal disordered residues removed;
    * `1ZUG_dimer2.pdb`, a single structure coming from the NMR ensemble (1ZUG.pdb) with the terminal disordered residues removed. Differ from 1ZUG_dimer1.pdb only by chain ID;
    * `OR1_unbound.pdb`, a structure of the OR1 operator in B-DNA conformation;
    * `3CRO_complex.pdb`, an X-RAY structure of the sought-for complex, to be used to evaluate the docking poses.

* `restraints`:
  * `ambig_prot_dna.tbl`, a file containing the ambiguous restraints for this docking scenario.

* `runs`, a folder with HADDOCK3 output, produced by the `protein-dna-basic.cfg` workflow and by the `alascan.cfg` workflow (see Bonus section). We will navigate relevant parts of this folder throughout the tutorial.

* `scripts`, a directory containing various scripts used in this tutorial

* `workflows`, a directory containing configuration file examples for HADDOCK3


---
# **Understanding the Ambiguous Interaction Restraints**

The Ambiguous Interaction Restraints (AIRs) are crucial to successful docking, as they guide the modelling process by biasing partners’ conformations and co-orientation towards a certain, hopefully correct conformation of the complex. AIRs consist of the residues located in the interface of a complex.


## **Visualisation of the interface**

Let’s visualise residues in the interface of the 3CRO.

In [ ]:
#@title Visualize our target complex (3CRO)
import py3Dmol
import gzip
import os

# Create a 3Dmol viewer object
view = py3Dmol.view(width=800, height=600)

# Specify the path to your PDB file (can be .pdb or .pdb.gz)
pdb_file_path = PROJECT_DIR / "pdbs" / "3CRO_complex.pdb"

# Check if the file exists
if not os.path.exists(pdb_file_path):
    print(f"Error: File not found at {pdb_file_path}")
else:
    # Check if the file is gzipped
    if str(pdb_file_path).endswith('.gz'):
        with gzip.open(pdb_file_path, 'rt') as f:
            view.addModel(f.read(), 'pdb')
    else:
        with open(pdb_file_path, 'r') as f:
            view.addModel(f.read(), 'pdb')

    view.setStyle({"chain": "B"},{'cartoon': {'color': 'white'}})
    view.setStyle({"chain": "A"},{'cartoon': {'color': 'blue'}})
    view.setStyle({"chain": "C"},{'cartoon': {'color': 'magenta'}})
    view.addStyle({"chain": "A", "resi": [28, 29, 30, 31, 32, 34, 35]}, {'cartoon':{'color':'red'}})
    view.addStyle({"chain": "C", "resi": [28, 29, 30, 31, 32, 34, 35]}, {'cartoon':{'color':'red'}})
    view.addStyle({"chain": "A", "resi": [28, 29, 30, 31, 32, 34, 35]}, {'stick':{'color':'red'}})
    view.addStyle({"chain": "C", "resi": [28, 29, 30, 31, 32, 34, 35]}, {'stick':{'color':'red'}})
    view.addStyle({"chain": "B", "resi": [4,5,6,7,13,14,15,16,17,18,22,23,24,25,31,32,33,34,35,36]}, {'cartoon':{'color':'orange'}})
    view.addStyle({"chain": "B", "resi": [4,5,6,7,13,14,15,16,17,18,22,23,24,25,31,32,33,34,35,36]}, {'stick':{'color':'orange'}})
    view.zoomTo()
    view.show()


Let's now visualize the NMR structure of CRO alone (PDB entry 1ZUG), which consists of an ensemble of 20 models.

In [ ]:
#@title Visualize the apo receptor (NMR structure ensemble)
import gzip
import os
# Create a 3Dmol viewer object
view = py3Dmol.view(width=800, height=600)

# Specify the path to your PDB file (can be .pdb or .pdb.gz)
pdb_file_path = './pdbs/1ZUG.pdb'

# Check if the file exists
if not os.path.exists(pdb_file_path):
    print(f"Error: File not found at {pdb_file_path}")
else:
    # Check if the file is gzipped
    if pdb_file_path.endswith('.gz'):
        with gzip.open(pdb_file_path, 'rt') as file:
            pdb_data = file.read()
    else:
        with open(pdb_file_path, 'r') as file:
            pdb_data = file.read()

    view.addModelsAsFrames(pdb_data, "pdb")

    view.setStyle({'cartoon': {'color': '#F4B942'}})
    view.addStyle({'resi': [28, 29, 30, 31, 32, 34, 35]},{'cartoon':{'color':'red'}})
    view.addStyle({'resi': [28, 29, 30, 31, 32, 34, 35]},{'stick':{'color':'red'}})
    view.animate({"loop": "forward", "reps": 0, "interval": 500})
    view.zoomTo()
    view.show()


__*How much does the conformation of the interacting region change in the provided ensemble? Is this the most flexible region of the protein?*__

<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
The conformation of the interacting region does not change in the ensemble, this region is not the most flexible. The most flexible region is the tail of the protein, and it appears to not interact with the DNA.
  </p>
</details>
<br>

__*Can you spot one region that varies a lot?*__

<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
  The C-terminal tail has various conformations in the different models, indicating it is disordered in the NMR structure, most likely because of lack of NMR NOE information. Actually, if you compare it to the refence complex above, this region folds into a short helix and forms a protein-protein interface upon DNA binding. This a type of conformational change that docking can not easily reproduce. For this reason, for docking purposes, it is better to remove that disordered tail prior do docking (see preparation of PDB files below).
  </p>
</details>
<br>


Let’s switch to the surface representation of a single model:

In [ ]:
#@title Visualize a single model in surface representation highlighting the binding site
import gzip
import os

view = py3Dmol.view(width=800, height=600)

pdb_file_path = "./pdbs/1ZUG.pdb"

if not os.path.exists(pdb_file_path):
    print(f"Error: File not found at {pdb_file_path}")

else:
    if pdb_file_path.endswith(".gz"):
        with gzip.open(pdb_file_path, "rt") as file:
            pdb_data = file.read()
    else:
        with open(pdb_file_path, "r") as file:
            pdb_data = file.read()

    view.addModel(pdb_data, "pdb")

    view.setStyle({'cartoon': {'color': 'white'}})
    view.addStyle({'stick': {'color': 'white'}})
    view.addStyle({'resi': [28, 29, 30, 31, 32, 34, 35]},{'cartoon':{'color':'red'}})
    view.addStyle({'resi': [28, 29, 30, 31, 32, 34, 35]},{'stick':{'color':'red'}})
    view.addSurface(py3Dmol.SAS, {'opacity':0.4, 'color':'white'})
    view.zoomTo()
    view.show()


Let's now look at the DNA input structure we will use for docking. It was built as a perfect B-DNA conformation.

In [ ]:
#@title Visualize the DNA
import py3Dmol
import gzip
import os

view = py3Dmol.view(width=800, height=600)

pdb_file_path = "./pdbs/OR1_unbound.pdb"

if not os.path.exists(pdb_file_path):
    print(f"Error: File not found at {pdb_file_path}")

else:
    if pdb_file_path.endswith(".gz"):
        with gzip.open(pdb_file_path, "rt") as file:
            pdb_data = file.read()
    else:
        with open(pdb_file_path, "r") as file:
            pdb_data = file.read()

    view.addModel(pdb_data, "pdb")

    view.setStyle({'cartoon': {'color': 'white'}})
    view.addStyle({'stick': {'color': 'white'}})
    view.addStyle({'resi': [4,5,6,7,13,14,15,16,17,18,22,23,24,25,31,32,33,34,35,36]},{'cartoon':{'color':'red'}})
    view.addStyle({'resi': [4,5,6,7,13,14,15,16,17,18,22,23,24,25,31,32,33,34,35,36]},{'stick':{'color':'red'}})
    view.addSurface(py3Dmol.SAS, {'opacity':0.4, 'color':'white'})
    view.zoomTo()
    view.show()


__*Compare this conformation to the one in the refence complex above? Can you see differences?*__

<details style="background-color:#DAE4E7">
  <summary style="bold">
    <i>Answer</i> <i class="material-icons">expand</i>
  </summary>
  <p>
  This conformation is a perfect B-DNA conformation, meaning perfectly straight. Looking that the complex above we can observe that upon CRO binding bending of the DNA occurs. This is thus a conformational change. It is a rather large change, which in this tutorial will be difficult to model. In our original protein-DNA docking paper, we described a two step docking procedure to induce larger conformational changes in the DNA. Check: van Dijk, M. et al. (2006). Information-driven protein-DNA docking using HADDOCK: it is a matter of flexibility. NAR. 25, 3317-3325 
  </p>
</details>
<br>


## **What’s inside the restraints file?**

The ambiguous interaction restraints are defined in the `ambig_prot_DNA.tbl` file in the `restraints` directory. This file was created using both experimental knowledge and information from literature. A detailed explanation of how to generate these restraints can be found in the advanced version of the HADDOCK2.4 tutorial, accessible [here](https://www.bonvinlab.org/education/HADDOCK24/HADDOCK24-protein-DNA-advanced/#available-data).

Let’s have a look at it’s first lines:

```
assign ( resid 35 and segid C)
(
   (  resid 32 and segid B and (name H3 or name O4 or name C4 or name C5 or name C6 or name C7))
    or
    ( resid 33 and segid B and (name H3 or name O4 or name C4 or name C5 or name C6 or name C7))
) 2.000 2.000 0.000
```

The first line means that the residue 35 from the chain C (protein) should interact either with the residue 32, or with the residue 33 of the chain B (DNA). Additionally, the substring `(name H3 or name O4 or name C4 or name C5 or name C6 or name C7)` precise the atoms with which the interaction should occur.

To simplify, if at least one pair or residues (residue 35 from chain C; residue 32 from chain B) or (residue 35 from chain C; residue 33 from chain B) are located in the vicinity from one another - then this particular restraint is satisfied.

__*Check out the list of atoms defined in `ambig_prot_DNA.tbl` in the extract above. Which part of the DNA is targeted by defining a given set of atoms? What is the effect of such restraints onto the docking process?*__

<details style="background-color:#DAE4E7">
  <summary style="bold">
    <i>Answer</i> <i class="material-icons">expand</i>
  </summary>
  <p>
  The atom selections on the DNA are pointing to nucleotide base atoms in the major groove. If we were to define only the residue number, contacts with the ribose or the phosphate would already satisty the restraints, but would not explain the specificity of CRO for this particular DNA sequence.
  </p>
</details>
<br>

You may notice that not all residues of the protein’s interface are used for the AIRs. This is done to save computational time.

HADDOCK is not limited to ambiguous restraints, other types, like unambiguous and symmetry restraints can play an important role as well. As mentioned before, Cro is known to function as a symmetrical dimer. This means we should __enforce a pairwise symmetry (C2)__ between the two protein monomers. This part will be explained in the [HADDOCK3 workflow](https://www.bonvinlab.org/education/HADDOCK3/HADDOCK3-protein-DNA-basic/#haddock3-workflow) section of the tutorial.

---
# **Preparation of PDB files for the docking** (optional)

Haddock3 requires an input structure for each docking partner. The quality of these input structures are highly influential to the quality of the docking models. Conformational deficiencies such as sterical clashes, chain breaks and missing atoms may cause problems during the docking, so it is important to verify each input file. Another important factor is the difference between unbound and bound conformations. The more different these conformations are, the more difficult it is to generate correct docking models.

In this section we will go over the preparation of the protein structures. The preparation of the DNA structure is out of the scope of this tutorial, but is detailed in the [advanced tutorial](https://www.bonvinlab.org/education/HADDOCK24/HADDOCK24-protein-DNA-advanced/#preparing-pdb-coordinate-files-for-the-or1-operator).

Ready-to-dock structures are available in `pdbs` directory, namely `1ZUG_dimer1.pdb`, `1ZUG_dimer2.pdb` and `OR1_unbound.pdb`.

## **Protein structures**

An unbound structure of the protein is available on [PDB](https://www.rcsb.org/structure/1ZUG). We already examined this structure using PyMOL. Our observation revealed that this protein has a disordered tail, which does not interact with the DNA. Since the core conformation remains unchanged, we can simply take the first conformation from the ensemble, remove the disordered tail from it and use it as an input structure for the docking

This can be done using `pdb-tools`, a collection of simple scripts handy to manipulate pdb files. `pdb-tools` is installed automatically with Haddock3. Alternatively, it is also available as a [web-server](https://wenmr.science.uu.nl/pdbtools/).

To obtain a single trimmed structure, we will make use of the command-line version of `pdb-tools`. Please, __*remember to activate a virtual environment for Haddock3*__ before using `pdb-tools`. The following command will override the existing file with the name `1ZUG_dimer1.pdb` - consider executing it outside of `pdbs/`:

1. This sequence of commands: 1/Downloads given structure in PDB format from the RCSB website; 2/ Extracts the first model from the file; 3/ Removes all HETATM records in the PDB file; 4/ Selects residues by their index (in a range); 5/ Adds TER statement at the end of the chain.


In [ ]:
! pdb_fetch 1ZUG | pdb_selmodel -1 | pdb_delhetatm | pdb_selres -1:66 | pdb_tidy -strict > 1ZUG_dimer1.pdb

The complex of interest contains 2 copies of the protein. As each molecule given to HADDOCK in a docking scenario must have a __unique chain ID and segment ID__, we have to change the chain ID from A to C and save this as a new structure. This can be achieved using another command from `pdb_tools`, namely, `pdb_rplchain`, which stands for “replace chain”:

In [ ]:
! pdb_rplchain -A:C 1ZUG_dimer1.pdb > 1ZUG_dimer2.pdb

__*Note*__ that it is possible to perform the docking with an ensemble of trimmed conformations. Such ensemble can be obtained using the following command: `pdb_fetch 1ZUG | pdb_delhetatm | pdb_selres -1:66 | pdb_tidy -strict > 1zug_ens-trimmed.pdb`. Here the command `pdb_selmodel -1` was removed and therefore all the available models in the ensemble will be processed. But if we are applying C2 symmetry, in the case of an ensemble, it is better to combine only the same conformations (to ensure identical conformations), meaning we will have to set `crossdock=false` in the rigidbody module. And also all input PDB files will have to contain the same number of models (which means multiple (as many as the NMR ensemble) copies of the B-DNA conformations will have to be combined into one PDB ensemble).

## **The DNA**

The DNA structure we will use (provided in the `pdbs` directory was build from the DNA sequence as a perfect B-form DNA conformation using at the time our 3D-DART server (no longer available).

---
# **Setting up and running the docking with HADDOCK3**

Now that we have all required files at hand (PDB and restraints files), it is time to setup our docking protocol.
In this tutorial, considering we have rather good information about the binding sites, we will execute a fast HADDOCK3 docking workflow, reducing the non-negligible computational cost of HADDOCK by decreasing the sampling, without impacting too much the accuracy of the resulting models.


## **Specifics of the protein-DNA docking**

Docking a double-stranded DNA requires adjusting several default parameters to better mimic the conditions under which DNA interactions occur. The following parameters should be modified:

* Add an automatic restraint to maintain the input conformation of the DNA during refinement: `dnarest_on = true`;
* Perform explicit solvent molecular dynamics refinement instead of energy minimization refinement by using the `[mdref]` module instead of `[emref]`;
* Set the dielectric constant to 78 for both sampling and flexible refinement (but not for MD refinement): `epsilon = 78`;
* Fix the relative dielectric constant in the Coulomb potential (rather than using a distance-dependent mode) for both sampling and flexible refinement: `dielec = cdie`;
* Set the weight of the desolvation energy term to 0: `w_desolv = 0`;
* Lower the scaling factor for flexible refinement to 4 (from 8) to allow less movement during the refinement: `tadfactor = 4`;
* Lower the initial temperature for the final round of flexible refinement to 300 (from 1000) to allow less movement during the final refinement stage: `temp_cool3_init = 300`.

__*Note*__ that HADDOCK3 distinguishes DNA nucleotides from RNA nucleotides based on the residue naming in the PDB file. DNA nucleotides are named with two letters starting with ‘D’ (e.g., ‘DA’ for deoxyadenosine in DNA), while RNA nucleotides use single-letter names (e.g., ‘A’ for adenosine in RNA).

## **HADDOCK3 workflow definition**

Now that we have all the necessary files ready for docking, along with several insights into the specifics of protein-DNA docking, it’s time to create the docking workflow. In this scenario, we will adhere to the following straightforward workflow: rigid-body docking, semi-flexible refinement in torsional angle space, molecular dynamics (MD) refinement in explicit solvent, and a final clustering step.

We will also integrate two analysis modules in our workflow:

- `caprieval` will be used at various stages to compare models to either the best scoring model (if no reference is given) or a reference structure, which in our case we have at hand (`pdbs/3CRO_complex.pdb`). This will directly allow us to assess the performance of the protocol. In the absence of a reference, `caprieval` is still useful to assess the convergence of a run and analyse the results.
- `contactmap` added as last module will generate contact matrices of both intra- and intermolecular contacts and a chordchart of intermolecular contacts for each cluster.

Our workflow consists of the following modules:

* __topoaa__: *Generates the topologies for the CNS engine and builds missing atoms;*
* __rigidbody__: *Performs sampling by rigid-body energy minimization (equivalent to `it0` in Haddock2.X);*
* __caprieval__: *Calculates CAPRI metrics (i-RMSD, l-RMSD, Fnat, DockQ) with respect to the best-scored model or a provided reference structure;*
* __seletop__: *Selects X best-scored models from the previous module;*
* __flexref__: *Performs semi-flexible refinement of the interface (equivalent to `it1` in Haddock2.X);*
* __caprieval__
* __mdref__: *Performs final refinement via explicit solvent MD (equivalent to itw in Haddock2.X);*
* __caprieval__
* __clustfcc__: *Fraction of contact based clustering;*
* __seletopclusts__: *Selects X best-scored models of Y clusters.*
* __caprieval__: *Final assessment of the docking results.*
* __contactmap__: *Generate a contact map for each cluster*

As mentioned before, we should enforce C2 symmetry between the proteins throughout the entire docking process. This can be easily achieved by adding the following parameters to the `[rigidbody]` , `[flexref]` , and `[mdref]` modules:

```toml
# Turn on symmetry restraints
sym_on = true
# Define first symmetry partner
c2sym_seg1_1 = 'A'
# Define second symmetry partner
c2sym_seg2_1 = 'C'
# Specify the range of residues that should be taken from the first partner
c2sym_sta1_1 = 4
c2sym_end1_1 = 64
# Specify the range of residues that should be taken from the second partner
c2sym_sta2_1 = 4
c2sym_end2_1 = 64
```
__Note__ that in this definition we omitted the first 3 residues of each protein.

The corresponding toml configuration file (provided in `workflows/protein-dna-basic.cfg`) looks like:

```toml
# ====================================================================
# Protein-DNA basic docking example with:
# 1. Pre-generated ambiguous restraints between protein dimer and DNA partners
# 2. Pairwise (C2) symmetry between the two protein monomers
# 3. Specific to double-stranded DNA-protein docking parameters
# ====================================================================

# directory in which the docking will be performed
run_dir = "run1"

# compute mode
mode = "local"
ncores=10

# input pdbs of the docking partners
molecules =  [
    "pdbs/1ZUG_dimer1.pdb",
    "pdbs/OR1_unbound.pdb",
    "pdbs/1ZUG_dimer2.pdb"
    ]

#compress all generated models
clean = true

# ====================================================================
# Workflow is defined as a pipeline of modules with specified parameters per module
# ====================================================================

[topoaa]

[rigidbody]
sampling = 100
# Cro to OR1 ambiguous restraints
ambig_fname = "restraints/ambig_prot_DNA.tbl"
# C2 symmetry
sym_on = true
c2sym_seg1_1 = 'A'
c2sym_seg2_1 = 'C'
c2sym_sta1_1 = 4
c2sym_sta2_1 = 4
c2sym_end1_1 = 64
c2sym_end2_1 = 64
# dielectric constant
epsilon = 78
# fix constant in Coulomb potential
dielec = 'cdie'
# turn off desolvation from the scoring function
w_desolv = 0

[caprieval]
reference_fname = "pdbs/3CRO_complex.pdb"

[seletop]
select = 20

[flexref]
# to maintain conformation of the DNA with automatic restraints
dnarest_on = true
# Cro to OR1 ambiguous restraints
ambig_fname = "restraints/ambig_prot_DNA.tbl"
# C2 symmetry
sym_on = true
c2sym_seg1_1 = 'A'
c2sym_seg2_1 = 'C'
c2sym_sta1_1 = 4
c2sym_sta2_1 = 4
c2sym_end1_1 = 64
c2sym_end2_1 = 64
# dielectric constant
epsilon = 78
# fix constant in Coulomb potential
dielec = 'cdie'
# turn off desolvation from the scoring function
w_desolv = 0
# reduced temperature for the final flexible refinement stage
temp_cool3_init = 300
# reduced time step
tadfactor = 4

[caprieval]
reference_fname = "pdbs/3CRO_complex.pdb"

[mdref]
# to maintain conformation of the DNA with automatic restraints
dnarest_on = true
# Cro to OR1 ambiguous restraints
ambig_fname = "restraints/ambig_prot_DNA.tbl"
# C2 symmetry
sym_on = true
c2sym_seg1_1 = 'A'
c2sym_seg2_1 = 'C'
c2sym_sta1_1 = 4
c2sym_sta2_1 = 4
c2sym_end1_1 = 64
c2sym_end2_1 = 64
# turn off desolvation from the scoring function
w_desolv = 0

[caprieval]
reference_fname = "pdbs/3CRO_complex.pdb"

[clustfcc]

[seletopclusts]

[caprieval]
reference_fname = "pdbs/3CRO_complex.pdb"

[contactmap]

# ====================================================================
```

__Note__ that in this example we reduced the sampling at rigidbody to 100 models and select only 20 models for flexible refinement in order to speed-up the calculations. A pre-computed full run with default settings is available in the `runs` directory as `run1-full`.

This workflow begins by creating topologies for the docking partners (`[topoaa]`). Rigid body sampling (`[rigidbody]`) is performed with ambiguous and symmetry restraints, generating 1000 models, from which the top 200 are selected (`[seletop]`). These models then undergo flexible refinement (`[flexref]`) followed by MD refinement in explicit solvent (`[mdref]`), still maintaining the same ambiguous and symmetry restraints. Finally, a Fraction of Common Contacts (FCC) based clustering is performed (`[clustfcc]`). The top 10 models from each cluster are selected (`[seletopclusts]`). The `caprieval` module is added after each step using the crystal structure as a reference to simplify the analysis and track the rank of models throughout the docking process. As last step we call the `contactmap` module to visualize the contacts.


**_Note_**: For making best use of the available CPU resources it is recommended to adapt the sampling parameter to be a multiple of the number of available cores when running in local mode.

**_Note_**: In case no reference is available (the usual scenario), the best ranked model is used as reference for each stage.
Including `caprieval` at the various stages even when no reference is provided is useful to get the rankings and scores and visualise the results (see Analysis section below).

**_Note_**: The default sampling would be 1000 models for `rigidbody` of which 200 are passed to the flexible refinement in `seletop`.
As an indication of the computational requirements, the default sampling worflow for this tutorial completes in about 40 min using 10 cores on a MaxOSX M2 processor. In comparison, the reduced sampling run (100/20) takes about 4-5min.

**_Note_**: To get a list of all possible parameters that can be defined in a specific module (and their default values) you can use the following command:

```
haddock3-cfg -m <module-name>
```
Add the `-d` option to get a more detailed description of parameters and use the `-h` option to see a list of arguments and options.

Alternatively, you can consult the [developer's guide](https://www.bonvinlab.org/haddock3/), where each parameter of each module is listed along with their default values, short and long descriptions, etc. Navigate to the [Modules](https://www.bonvinlab.org/haddock3/modules/index.html#) and select a module which parameters you want to display.



---

## Running HADDOCK3

In the first section of the configuration file you can see the definition of the global parameters:

```toml
# compute mode
mode = "local"
ncores = 10
```
The parameter `mode` defines how this workflow will be executed. In this case, it will run locally, on your machine, using up to 10 CPUs. Feel free to change this value, if more cores are available.

To execute the above workflow you have two choices:

1. __Open a terminal session__ and simply type (check if the location is correct):

```
cd HADDOCK3-protein-dna-notebook
haddock3 workflows/protein-dna-basic.cfg >&run.out &
```

This will run haddock3 in background. In the mean time we can contine with the analysis section using the precalculated results.

You can check the status of the run by looking at the `run.out` file.

2. __Or execute the next cell__, and wait for its completion to proceed with the analysis (time for a cup of coffee :-) as the run should take about 15 minutes to complete)

In [ ]:
!haddock3 workflows/protein-dna-basic.cfg



In case the execution was interupted you can restart haddock3 from the last completed stage. Check the content of the run1 directory and find out which stage was the last running and not completed. Then restart haddock3 with as addional option: ``--restart 4`` (for example to restart at stage 4). This can be done from the terminal within the `HADDOCK3-protein-dna-notebook` directory.

```
haddock3 workflows/protein-dna-basic.cfg --restart 4
```

---
# **Analysis of the docking results**

Let’s get acquainted with the contents of the resulting directory - take a look inside. You will find that the various steps of our workflow (modules) are numbered sequentially, starting at 0:

```
> ls run1/
    00_topoaa/
    01_rigidbody/
    02_caprieval/
    03_seletop/
    04_flexref/
    05_caprieval/
    06_mdref/
    07_caprieval/
    08_clustfcc/
    09_seletopclusts/
    10_caprieval/
    11_contactmap/
    analysis/
    data/
    log
    traceback/
```

Additionally, you will find the following directories and files:

* `log` file: This file allows you to verify the execution of each module. More importantly, if the docking run fails, you can identify the cause by carefully reading the error messages. Also, at the very bottom of the file, you can see the total execution time of the docking run;
* `analysis` directory: Contains information relevant to the results of each caprieval step, including file report.html with thee table containing limited number of the top-ranked models/clusters and various plots to visualise statistics of the results;
* `data` directory: Stores input PDB files (not the actual docking models) and restraints files used in each module;
* `traceback` directory: Contains the `traceback.tsv` file, which tracks the name and rank of each docking model throughout the entire workflow. Handy, if you want to see how the model’s rank evolved step-by-step.

Each sampling, refinement, or selection module’s directory contains compressed PDB files of the docking models. For example, `09_seletopclusts` contains 10 top-ranked docking models from each of the 4 clusters. Clusters are numbered based on the average rank of the models within them, so cluster_1 contains the top-ranked models of the entire run. Information about the origin of each model can be found in `09_seletopclusts/seletopclusts.txt`, as well as in `traceback/traceback.tsv`.


The simplest way to extract ranking information and the corresponding HADDOCK scores is to look at the `XX_caprieval` directories (which is why it is a good idea to have it as the final module, and possibly as intermediate steps). This directory will always contain a `capri_ss.tsv` single model statistics file, which contains the model names, rankings and statistics (score, iRMSD, Fnat, lRMSD, ilRMSD and dockq score). E.g. for `10_caprieval`:

```
model	md5	caprieval_rank	score	irmsd	fnat	lrmsd	ilrmsd	dockq	rmsd	cluster_id	cluster_ranking	model-cluster_ranking	air	angles	bonds	bsa	cdih	coup	dani	desolv	dihe	elec	improper	rdcs	rg	sym	total	vdw	vean	xpcs
../09_seletopclusts/cluster_1_model_1.pdb	-	1	-184.763	5.978	0.435	16.105	14.775	0.237	6.552	1	1	1	47.509	1644.740	44.990	2164.950	0.000	0.000	0.000	24.284	701.425	-670.437	35.785	0.000	0.000	3.257	-656.349	-55.426	0.000	0.000
../09_seletopclusts/cluster_1_model_2.pdb	-	2	-182.602	4.141	0.324	13.299	12.154	0.243	4.427	1	1	2	79.534	1642.770	44.663	2350.730	0.000	0.000	0.000	22.037	719.080	-588.207	37.119	0.000	0.000	2.322	-561.687	-72.914	0.000	0.000
../09_seletopclusts/cluster_1_model_3.pdb	-	3	-181.407	5.354	0.444	16.959	15.405	0.239	5.866	1	1	3	34.958	1642.750	45.145	2314.340	0.000	0.000	0.000	17.422	714.709	-585.596	37.944	0.000	0.000	2.943	-598.484	-67.783	0.000	0.000
../09_seletopclusts/cluster_1_model_4.pdb	-	4	-181.168	3.379	0.398	10.262	9.519	0.323	3.634	1	1	4	57.679	1650.190	45.620	2213.770	0.000	0.000	0.000	19.697	722.757	-608.571	38.164	0.000	0.000	2.383	-596.624	-65.222	0.000	0.000
../09_seletopclusts/cluster_1_model_5.pdb	-	5	-179.463	5.246	0.417	17.059	15.468	0.230	5.698	1	1	5	69.673	1650.520	44.969	2189.320	0.000	0.000	0.000	18.131	729.287	-592.340	38.049	0.000	0.000	2.779	-571.017	-67.963	0.000	0.000
../09_seletopclusts/cluster_1_model_6.pdb	-	6	-179.323	5.247	0.380	15.857	14.312	0.226	5.776	1	1	6	30.837	1643.540	45.676	2137.900	0.000	0.000	0.000	19.036	712.655	-622.587	37.403	0.000	0.000	2.724	-630.011	-57.889	0.000	0.000
../09_seletopclusts/cluster_1_model_7.pdb	-	7	-178.906	8.508	0.361	23.311	21.262	0.170	9.309	1	1	7	31.553	1639.230	44.992	2135.660	0.000	0.000	0.000	16.619	727.419	-631.677	36.944	0.000	0.000	2.976	-635.779	-55.726	0.000	0.000
...
```

If clustering was performed prior to calling the `caprieval` module, the `capri_ss.tsv` file will also contain information about to which cluster the model belongs to and its ranking within the cluster.

The relevant statistics are:

* **score**: *the HADDOCK score (arbitrary units)*
* **irmsd**: *the interface RMSD, calculated over the interfaces the molecules*
* **fnat**: *the fraction of native contacts*
* **lrmsd**: *the ligand RMSD, calculated on the ligand after fitting on the receptor (1st component)*
* **ilrmsd**: *the interface-ligand RMSD, calculated over the interface of the ligand after fitting on the interface of the receptor (more relevant for small ligands for example)*
* **dockq**: *the DockQ score, which is a combination of irmsd, lrmsd and fnat and provides a continuous scale between 1 (exactly equal to reference) and 0*

Various other terms are also reported including:

* **bsa**: *the buried surface area (in squared angstroms)*
* **elec**: *the intermolecular electrostatic energy*
* **vdw**: *the intermolecular van der Waals energy*
* **desolv**: *the desolvation energy*


The iRMSD, lRMSD and Fnat metrics are the ones used in the blind protein-protein prediction experiment [CAPRI](https://www.ebi.ac.uk/msd-srv/capri/) (Critical PRediction of Interactions).

In CAPRI the quality of a model is defined as (for protein-protein complexes):

* **Unacceptable**:    i-RMSD >4Å or Fnat < 0.1 (DOCKQ < 0.23)
* **acceptable model**: i-RMSD < 4Å or l-RMSD < 10Å and Fnat > 0.1 (0.23 < DOCKQ < 0.49)
* **medium quality model**: i-RMSD < 2Å or l-RMSD < 5Å and Fnat > 0.3 (0.49 < DOCKQ < 0.8)
* **high quality model**: i-RMSD < 1Å or l-RMSD < 1Å and Fnat > 0.5 (DOCKQ > 0.8)

__*Based on these CAPRI criteria, what is the quality of the best model listed above (_cluster_1_model_1.pdb_)?*__


In case where the `caprieval` module is called after a clustering step, an additional `capri_clt.tsv` file will be present in the directory.
This file contains the cluster ranking and score statistics, averaged over the minimum number of models defined for clustering
(4 by default), with their corresponding standard deviations. E.g.:

```
cluster_rank	cluster_id	n	under_eval	score	score_std	irmsd	irmsd_std	fnat	fnat_std	lrmsd	lrmsd_std	dockq	dockq_std	ilrmsd	ilrmsd_std	rmsd	rmsd_std	air	air_std	bsa	bsa_std	desolv	desolv_std	elec	elec_std	total	total_std	vdw	vdw_std	caprieval_rank
1	1	10	-	-182.485	1.423	4.713	1.015	0.400	0.047	14.156	2.625	0.261	0.036	12.963	2.332	5.120	1.151	54.920	16.332	2260.948	74.754	20.860	2.563	-613.203	34.220	-603.286	33.962	-65.336	6.357	1
2	2	10	-	-173.774	4.243	5.279	0.720	0.354	0.030	14.924	2.062	0.228	0.029	14.532	1.932	5.560	0.713	94.918	25.476	2312.925	68.000	16.405	1.812	-519.811	18.712	-484.098	15.639	-79.304	4.829	2
3	3	10	-	-159.355	11.361	12.030	0.100	0.093	0.009	51.304	0.511	0.045	0.003	48.250	0.519	12.492	0.069	121.744	46.567	2227.635	65.050	13.737	1.889	-443.701	29.415	-384.879	56.769	-82.789	5.340	3
4	4	8	-	-157.864	12.488	9.008	3.046	0.273	0.045	24.093	6.927	0.147	0.041	22.095	6.364	9.780	3.349	63.992	8.718	2126.375	69.337	18.781	3.044	-503.914	48.940	-483.716	50.058	-63.481	4.785	4
5	6	4	-	-155.989	7.307	4.893	0.396	0.273	0.035	13.875	1.236	0.212	0.014	13.056	0.884	5.269	0.451	132.205	16.911	2222.088	65.201	18.111	3.872	-498.132	32.292	-414.972	40.532	-69.583	4.607	5
6	5	5	-	-134.460	3.162	13.234	0.891	0.095	0.004	56.010	2.134	0.043	0.001	53.010	2.012	14.004	1.039	107.635	12.232	2093.977	60.298	14.271	4.853	-409.716	37.169	-345.567	39.946	-63.280	7.915	6
```

In this file you find the cluster rank (which corresponds to the naming of the clusters in the previous `seletop` directory), the cluster ID (which is related to the size of the cluster, 1 being always the largest cluster), the number of models (n) in the cluster and the corresponding statistics (averages + standard deviations). The corresponding cluster PDB files will be found in the preceeding `09_seletopclusts` directory.

While these simple text files can be easily checked from the command line already, they might be cumbersome to read.
For that reason, we have developed a post-processing analysis that automatically generates html reports for all `caprieval` steps in the workflow.
These are located in the respective `analysis/XX_caprieval` directories and can be viewed using your favorite web browser.




---

## **Cluster statistics**

Let us now analyse the docking results. Use for that either your own run or a pre-calculated run provided in the `runs` directory.
The `analysis/10_caprieval_analysis` directory of the respective run directory contains various html files which can be visualised in a web browser. The `report.html` contains interactive plots that may take some time to generate.

For this tutorial we will visualise parts of this file directly in the notebook.

On the top of the page, you will see a table that summarises the cluster statistics (taken from the `capri_clt.tsv` file).
The columns (corresponding to the various clusters) are sorted by default on the cluster rank, which is based on the HADDOCK score.

For the sake of this tutorial we will analyse the results from the full sampling run provided in `runs/run1-full`.


In [ ]:
#@title View the cluster statistics table
import json
import re
from plotly.offline import iplot
from IPython.display import display
import pandas as pd

# Specify the path to your HTML file
html_file_path = PROJECT_DIR / "runs" / "run1-full" / "analysis" / "10_caprieval_analysis/report.html"

# Read the HTML file
with open(html_file_path, 'r') as f:
    html_content = f.read()

# Use a regular expression to find the script tag with id="datatable2" and extract its content
match_table = re.search(r'<script id="datatable2" type="application/json">\s*(.*?)\s*</script>', html_content, re.DOTALL)

if match_table:
    json_data_str_table = match_table.group(1)
    # Load the JSON data
    table_data = json.loads(json_data_str_table)

    # Extract the 'clusters' list from the JSON data
    clusters_data = table_data.get('clusters', [])

    # Create a pandas DataFrame from the clusters data
    df = pd.json_normalize(clusters_data)

    # Drop the columns related to best cluster files
    cols_to_drop = ['under_eval', 'total', 'rmsd', 'best1', 'best2', 'best3', 'best4', 'ilrmsd.mean', 'ilrmsd.std', 'air.mean', 'air.std', 'desolv.mean', 'desolv.std', 'elec.mean', 'elec.std', 'vdw.mean', 'vdw.std'] # Add more if there are more 'best' columns
    df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

    # Display the DataFrame
    display(df)
else:
    print("Could not find the script tag with id='datatable2' in the HTML file.")


Inspect the final cluster statistics.

__*How many clusters have been generated?*__

__*Look at the score of the first few clusters: Are they significantly different if you consider their average scores and standard deviations?*__

Since for this tutorial we have at hand the crystal structure of the complex, we provided it as reference to the `caprieval` modules.
This means that the iRMSD, lRMSD, Fnat and DockQ statistics report on the quality of the docked model compared to the reference crystal structure. Remember that high DockQ and FCC values, along with low RMSD values, indicate better model quality.

__*Note*__ that our docking is a three-body docking and all interfaces are considered when calculating the CAPRI metrics.

CAPRI is using the following criteria to assess the quality of a model:

* **Unacceptable**:    i-RMSD >4Å or Fnat < 0.1 (DOCKQ < 0.23)
* **acceptable model**: i-RMSD < 4Å or l-RMSD < 10Å and Fnat > 0.1 (0.23 < DOCKQ < 0.49)
* **medium quality model**: i-RMSD < 2Å or l-RMSD < 5Å and Fnat > 0.3 (0.49 < DOCKQ < 0.8)
* **high quality model**: i-RMSD < 1Å or l-RMSD < 1Å and Fnat > 0.5 (DOCKQ > 0.8)

__*How many clusters of acceptable or better quality have been generate according to CAPRI criteria?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
  Looking at the DockQ values, only the first two clusters has an average >0.23.
  </p>
</details>
<br>

__*What is the rank of the best cluster generated?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
  This is cluster1, i.e. the top-ranked cluster with an average DockQ values of 0.26.
  </p>
</details>
<br>

__*What is the cluster ID of the best cluster generated?*__
<details style="background-color:#DAE4E7">
  <summary style="bold">
      <b><i>Answer</i></b>
  </summary>
  <figure style="text-align: justified;">
  <p>
   The cluster ID is related to the size of the cluster with clusterID 1 being the largest one. In this case cluster1, the top-ranked cluster, has ID 1 and correspond to the largest cluster with a population of 127 members (this info can be found in the clustfcc step output in the run1-full/08_clustfcc/clustfcc.txt file.
  </p>
</details>
<br>


## **Visualisation of the HADDOCK scores and their components**

Below the cluster statistics table, you’ll find a series of plots displaying the HADDOCK score and its components against various metrics (i-RMSD, l-RMSD, FCC, DockQ), with clusters represented using color coding. The last rows show plots of cluster statistics, i.e. distributions of values per cluster, ordered by their HADDOCK score.

These plots are interactive. A menu will appear at the top right, just above the last plot in the first row, when you hover your mouse over it. This menu allows you to zoom in and out of the plots and toggle the visibility of clusters.

To view theses plots in the notebook execute the following cell.

__*Examine the plots (remember here that higher DockQ values and lower i-RMSD values correspond to better models)*__


In [ ]:
#@title View the various statistics plots
import json
import re
from IPython.display import display
import pandas as pd

# Specify the path to your HTML file
html_file_path = PROJECT_DIR  / "runs" / "run1-full" / "analysis" / "10_caprieval_analysis/report.html"

# Read the HTML file
with open(html_file_path, 'r') as f:
    html_content = f.read()

# Use a regular expression to find the script tag with id="data1" and extract its content
match_plot = re.search(r'<script id="data1" type="application/json">\s*(.*?)\s*</script>', html_content, re.DOTALL)

if match_plot:
    json_data_str_plot = match_plot.group(1)
    # Load the JSON data
    plot_data = json.loads(json_data_str_plot)

    # Set width and height to None to make the plot responsive
    if 'layout' in plot_data:
        plot_data['layout']['width'] = None
        #plot_data['layout']['height'] = None

    # Display the plot using iplot
    iplot(plot_data)
else:
    print("Could not find the script tag with id='data1' in the HTML file.")

__*Inspect the plots. Which of the score components correlates best with the quality of the models?*__

Depending on the docking models, there could be a set of unclustered models. It will be explicitly shown in `report.html` as ‘other’. You can see the origins of these models in `traceback/traceback.tsv`.


Finally, the report also shows plots of the cluster statistics (distributions of values per cluster ordered according to their HADDOCK rank).

To view those distribution plots execute the next cell.



In [ ]:
#@title View the various distribution plots
import json
import re
from plotly.offline import iplot

html_file_path = PROJECT_DIR / "runs" / "run1-full" / "analysis" / "10_caprieval_analysis" / "report.html"

with open(html_file_path, 'r') as f:
    html_content = f.read()

match_plot = re.search(r'<script id="data2" type="application/json">\s*(.*?)\s*</script>', html_content, re.DOTALL)

if match_plot:
    json_data_str_plot = match_plot.group(1)
    plot_data = json.loads(json_data_str_plot)

    if 'layout' in plot_data:
        plot_data['layout']['width'] = None
        # Remove the template entirely — it contains deprecated trace types
        # (e.g. heatmapgl) that newer plotly versions no longer recognize
        plot_data['layout'].pop('template', None)

    iplot(plot_data, validate=False)
else:
    print("Could not find the script tag with id='data1' in the HTML file.")



---

## **Some single structure analysis**


Going back to command line analysis, we are providing in the `scripts` directory a simple script that extracts some model statistics for acceptable or better models from the `caprieval` steps.
To use it, simply call the script with as argument the run directory you want to analyze, e.g.:

```
scripts/extract-capri-stats-dockq.sh runs/run1-full
```

_**Note**_ that this kind of analysis only makes sense when we know the reference complex and for benchmarking / performance analysis purposes.



In [ ]:
!scripts/extract-capri-stats-dockq.sh runs/run1-full


Look at the single structure statistics provided by the script.

__*How does the quality of the best model changes after flexible refinement? Consider here the various metrics.*__

<details style="background-color:#DAE4E7">
  <summary style="bold">
    <i>Answer</i> <i class="material-icons">expand</i>
  </summary>
  <p>
  We only observe small differences in the best model for the various metrics.
  The fraction of native contacts however is improving much more after flexible refinement (07_caprieval step).
  All this will of course depend on how different are the bound and unbound conformations and the amount of data used to drive the docking process.
  In general, from our experience, the more and better data at hand, the larger the conformational changes that can be induced.
  </p>
</details>
<br>

__*Is the best model always ranked first?*__

<details style="background-color:#DAE4E7">
  <summary style="bold">
    <i>Answer</i> <i class="material-icons">expand</i>
  </summary>
  <p>
    This is not the case. The scoring function is not perfect, but does a reasonable job at ranking models of acceptable or better quality on top in this case, especially if one considers the Fnat metric.
  </p>
</details>
<br>

**_Note_**: A similar script to extract cluster statistics is available in the `scripts` directory as `extract-capri-stats-dockq-clt.sh`.



---
# **Contact Analysis**

We have added a contact analysis module to HADDOCK3 that generates for each cluster both a contact matrix of the entire system showing all contacts within a 4.5Å cutoff and a chord chart representation of intermolecular contacts.

In the current workflow we run, those files can be found in the `11_contactmap` directory. These are again html files with interactive plots (hover with your mouse over the plots).

This file taken from the pre-computed run can be visualized by executing the next cell which will visualize the contacts of the first ranked cluster (cluster 1).

__*Can you identify which residue(s) make(s) the most intermolecular contacts?*__



In [ ]:
#@title View the contact chord chart
import json
import re
from plotly.offline import iplot
from IPython.display import display

# Specify the path to the HTML file containing the chord chart
html_file_path = PROJECT_DIR / "runs" / "run1-full" / "11_contactmap" / "cluster2_chordchart.html"

# Read the HTML file
with open(html_file_path, 'r') as f:
    html_content = f.read()

# Use a regular expression to find the script tag containing the plot data
# The id of the script tag might vary, so a more general pattern is used
# We assume the plot data is within a script tag with type="application/json"
match_plot = re.search(r'<script.*?type="application/json".*?>\s*(.*?)\s*</script>', html_content, re.DOTALL)

if match_plot:
    json_data_str_plot = match_plot.group(1)
    # Load the JSON data
    plot_data = json.loads(json_data_str_plot)
    # Display the plot using iplot
    iplot(plot_data)
else:
    print(f"Could not find the plot data in the HTML file at {html_file_path}")

---
# **Visualisation of the docking models**

It’s time to visualise some of the docking models. First let’s take a look at `cluster_1_model_1.pdb.gz`, the best-ranked model and the reference structure `3CRO_complex.pdb`.

### **Comparison to the crystal structure of the complex**

We will align the best model onto the crystal structure of the complex in three different ways as we are dealing with a three molecules docking problem:

1) Using all three chains
2) Using only chainA (the first protein) and the DNA (chain B)
3) Using only chainC (the second receptor) and the DNA (chain B)

Execute all three following cells and compare the RMSDs.

In [ ]:
from haddock.libs.libnotebooks import align_full
#@title View the top ranking model from the best cluster
#@markdown It will be shown superimposed onto the reference structure fitted using all three chains
# Define paths to PDB files to be aligned
# Here we are overlaying the reference and 1st model from the 1st cluster
path1 = PROJECT_DIR / "pdbs" / "3CRO_complex.pdb"
path2 = PROJECT_DIR / "runs" / "run1" / "09_seletopclusts" / "cluster_1_model_1.pdb.gz"

# Align molecules by chain and display the result.
# For the full alignement put all chain IDs in the `chains` variable.
align_full(str(path1), str(path2), chains = ['A','B','C'])

In [ ]:
from haddock.libs.libnotebooks import align_full
#@title View the top ranking model from the best cluster
#@markdown It will be shown superimposed onto the reference structure fitted only on the 1st receptor and the DNA
# Define paths to PDB files to be aligned
# Here we are overlaying the reference and 1st model from the 1st cluster
path1 = PROJECT_DIR / "pdbs" / "3CRO_complex.pdb"
path2 = PROJECT_DIR / "runs" / "run1" / "09_seletopclusts" / "cluster_1_model_1.pdb.gz"

# Align molecules by chain and display the result.
# For the full alignement put all chain IDs in the `chains` variable.
align_full(str(path1), str(path2), chains = ['A','B'])

In [ ]:
from haddock.libs.libnotebooks import align_full
#@title View the top ranking model from the best cluster
#@markdown It will be shown superimposed onto the reference structure fitted only on the second receptor and the DNA
# Define paths to PDB files to be aligned
# Here we are overlaying the reference and 1st model from the 1st cluster
path1 = PROJECT_DIR / "pdbs" / "3CRO_complex.pdb"
path2 = PROJECT_DIR / "runs" / "run1" / "09_seletopclusts" / "cluster_1_model_1.pdb.gz"

# Align molecules by chain and display the result.
# For the full alignement put all chain IDs in the `chains` variable.
align_full(str(path1), str(path2), chains = ['B','C'])

__*Compare the global RMSDs and also look at the models: How well defined at the individual interfaces compared to the overall complex?*__

<details style="background-color:#DAE4E7">
  <summary style="bold">
    <i>Answer</i> <i class="material-icons">expand</i>
  </summary>
  <p>
  From a comparion of global RMSD values we can see that the individual interfaces are rather well defined (remember we started from an free receptor structure and a DNA in perfect B-DNA conformation), while the overall complex shows higher deviations.
  </p>
</details>
<br>


__*Which of the two interfaces is better defined?*__

<details style="background-color:#DAE4E7">
  <summary style="bold">
    <i>Answer</i> <i class="material-icons">expand</i>
  </summary>
  <p>
  Comparing the global RMSDs when fitting on the separate receptor, the first interface is better defined.
  </p>
</details>
<br>


Now let's look at `cluster_1_model_4.pdb.gz`, the model with the lowest i-RMSD (as shown in the plot “HADDOCK score vs i-RMSD”), and the reference structure `3CRO_complex.pdb`.

In [ ]:
from haddock.libs.libnotebooks import align_full
#@title View the lowest iRMSD model from the best cluster
#@markdown It will be shown superimposed onto the reference structure fitted using all three chains
# Define paths to PDB files to be aligned
# Here we are overlaying the reference and 1st model from the 1st cluster
path1 = PROJECT_DIR / "pdbs" / "3CRO_complex.pdb"
path2 = PROJECT_DIR / "runs" / "run1" / "09_seletopclusts" / "cluster_1_model_4.pdb.gz"

# Align molecules by chain and display the result.
# For the full alignement put all chain IDs in the `chains` variable.
align_full(str(path1), str(path2), chains = ['A','B','C'])

*Note* that models are compressed because of the line `clean = true` in the global parameters of the workflow. To decompress all models in a directory, one can run `haddock3-unpack <path/to/directory>`

As a final note, the free form receptor NMR structure we used for docking is missing the C-terminal helix which is found at the interface of the two receptors as this part only fold upon DNA binding. This can explain some of the differences in the relative orientation of the two receptors.

---
# **Congratulations!**

You’ve reached the end of this basic protein-DNA docking tutorial. We hope it has been informative and helps you get started with your own docking projects. Check out the advanced version of this tutorial ([currently available only for Haddock2.4 server](https://www.bonvinlab.org/education/HADDOCK24/HADDOCK24-protein-DNA-advanced/)) for deeper insights into protein-DNA docking!

Happy docking!

---
---
# **BONUS**

---
## **Alanine Scanning module**

A way of exploring interface energetics is by using the `alascan` module of HADDOCK3. `alascan` stands for "Alanine Scanning module".

This module is capable of mutating interface residues to Alanine and calculating the **Δ HADDOCK score** between the wild-type and mutant, thus providing a measure of the impact of each individual mutation. It is possible to scan all interface residues one by one or limit this scanning to a selected by user set of residues. By default, the mutation to Alanine is performed, as its side chain is just a methyl group, so side chain perturbations are minimal, as well as possible secondary structure changes. It is possible to perform the mutation to any other amino acid type - at your own risk, as such mutations may introduce structural uncertainty.

**Important**: 1/ `alascan` calculates the difference between wild-type score vs mutant score, i.e. positive `Δscore` indicative of the enriched (stronger) binding and negative `Δscore` is indicative of the depleted (weaker) binding; 2/ Inside `alascan`, a short energy minimization of an input structure is performed, i.e. there's no need to include an additional refinement module prior to `alascan`.


**Here's the workflow in `contactmap.cfg`**

```toml
# ====================================================================
# Alanine scanning of an interface
# ====================================================================

# directory in which the scoring will be done
run_dir = "run-alascan"

# compute mode
mode = "local"
ncores = 10

# Post-processing to generate statistics and plots
postprocess = true
clean = true

molecules =  [
    "pdbs/3CRO_complex.pdb",
    ]

# ====================================================================
# Parameters for each stage are defined below
# ====================================================================
[topoaa]

[emscoring]

[alascan]
plot = true
scan_residue = 'ALA'

# ====================================================================
```

An alanine scanning scenario configuration file is provided in the `workflows` directory as `interaction-energetics.cfg`, and precomputed results are in `runs/run-alascan`.


In [ ]:
#@title Run the haddock3 alanine scanning workflow
!haddock3 workflows/interaction-energetics.cfg



The output folder contains, among others, a directory titled `2_alascan` with a file `scan_clt_unclustered.tsv` that lists each mutation, corresponding score and individual terms:

```
################################################################################
# `alascan` cluster results for cluster unclustered
# reported values are the average for the cluster
#
# z_score is calculated with respect to the mean values of all residues
################################################################################
chain   resid   resname full_resname    delta_score     delta_score_std delta_vdw       delta_vdw_std   delta_elec      delta_elec_std  delta_desolv    delta_desolv_std        delta_bsa       delta_bsa_std   frac_pres       z_score
A	9	LYS	A-9-LYS	-13.70	0.00	0.45	0.00	-79.84	0.00	1.82	0.00	32.65	0.00	1.00	-1.23
A	12	ARG	A-12-ARG	-12.15	0.00	1.43	0.00	-71.62	0.00	0.74	0.00	-8.13	0.00	1.00	-0.97
A	18	THR	A-18-THR	-2.54	0.00	-0.79	0.00	-15.51	0.00	1.35	0.00	7.54	0.00	1.00	0.63
A	19	GLN	A-19-GLN	-5.95	0.00	-2.92	0.00	-11.41	0.00	-0.75	0.00	-17.50	0.00	1.00	0.06
A	20	THR	A-20-THR	0.41	0.00	-0.51	0.00	1.62	0.00	0.60	0.00	-1.17	0.00	1.00	1.12
A	28	VAL	A-28-VAL	-0.80	0.00	-1.07	0.00	0.21	0.00	0.23	0.00	-2.94	0.00	1.00	0.92
A	29	LYS	A-29-LYS	-13.56	0.00	2.42	0.00	-98.92	0.00	3.81	0.00	66.54	0.00	1.00	-1.20
...
```


Let us visualise the impact of each mutated residue for the best ranked cluster (cluster 1). This can be done by looking at the interactive plots generated in the 5_alascan directory

In [ ]:
#@title Visualise the results of the alanine scanning
import json
import re
from plotly.offline import iplot
from IPython.display import display

# Specify the path to the HTML file
html_file_path = PROJECT_DIR / "runs" / "run-alascan" / "2_alascan" / "scan_clt_unclustered.html"

# Read the HTML file
try:
    with open(html_file_path, 'r') as f:
        html_content = f.read()

    # Use a regular expression to find a script tag with type="application/json" and extract its content
    # The id of the script tag might vary, so a general pattern is used
    match_plot = re.search(r'<script id="data1" type="application/json">\s*(.*?)\s*</script>', html_content, re.DOTALL)

    if match_plot:
        json_data_str_plot = match_plot.group(1)
        # Load the JSON data
        plot_data = json.loads(json_data_str_plot)

        # Set width and height to None to make the plot responsive
        if 'layout' in plot_data:
            plot_data['layout']['width'] = None
            plot_data['layout']['height'] = None

        # Display the plot using iplot
        iplot(plot_data)
    else:
        print("Could not find JSON plot data in the HTML file.")

except FileNotFoundError:
    print(f"Error: File not found at {html_file_path}")
except Exception as e:
    print(f"An error occurred: {e}")

__*Can you identify the most enriching/depleting mutation of each chain?*__

You can use an additional script /scripts/get-alascan-extrema.sh to check your answer:


In [ ]:
!bash scripts/get-alascan-extrema.sh runs/run-alascan/2_alascan/scan_clt_unclustered.tsv

Mutation of the residue PHE48 in both receptor chains turned out to be the most depleting mutation, which indicate this residues has a very important contribution to the binding. Let's visualise it!

In [ ]:
#@title Visualise the Phe48 environment
import py3Dmol
import gzip
import os

# Create a 3Dmol viewer object
view = py3Dmol.view(width=800, height=600)

# Specify the path to your PDB file (can be .pdb or .pdb.gz)
pdb_file_path = PROJECT_DIR / "pdbs" / "3CRO_complex.pdb"

# Check if the file exists
if not os.path.exists(pdb_file_path):
    print(f"Error: File not found at {pdb_file_path}")
else:
    # Check if the file is gzipped
    if str(pdb_file_path).endswith('.gz'):
        with gzip.open(pdb_file_path, 'rt') as f:
            view.addModel(f.read(), 'pdb')
    else:
        with open(pdb_file_path, 'r') as f:
            view.addModel(f.read(), 'pdb')

    view.setStyle({'cartoon': {'color': 'white'}})
    view.addStyle({'stick': {}})
    view.addStyle({'resi': '48', 'chain': 'A'},{'sphere':{}})
    view.addStyle({'resi': '48', 'chain': 'C'},{'sphere':{}})
    view.addSurface(py3Dmol.SAS, {'opacity':0.4, 'color':'white'})
    view.zoomTo()
    view.show()

Phe48 is actually not contacting the DNA, but rather critical residue to stabilise the interface between the two receptors!

__*From the alascan plot above try to identify other important residues that might be involved in the DNA binding and visualise them by changing the residue number in the visualisation script.*__

<details style="background-color:#DAE4E7">
  <summary style="bold">
    <i>Answer</i> <i class="material-icons">expand</i>
  </summary>
  <p>
  Some of the other residues with a large contribution to the binding are LYS29, ARG43 and ARG45.
  <li> LYS29 seems to form a salt-bridge with the DNA phosphate</li>
  <li> ARG43 seems to be rather involved again in inter-receptor contacts.</li>
  <li> ARG45 penetrates deep into the minor groove of the DNA</li>
  </p>
</details>
<br>



---

---

# **Save your results to Google Drive**

In order to save the entire tutorial directory with your results run the following cell.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the source directory
source_dir = './HADDOCK3-protein-dna-notebook'

# Define the destination directory in Google Drive
destination_dir = '/content/drive/MyDrive/'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Move the directory
!cd ../; mv "$source_dir" "$destination_dir"

print(f"Moved '{source_dir}' to '{destination_dir}'")